<a href="https://colab.research.google.com/github/pop123-ux/Qwen2.5-7B-Instruct_finetuned_for_seedance2.0_prompting/blob/main/cdo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import requests
import io

# Updated URL to directly resolve the raw file content, including Git LFS files
url = "https://huggingface.co/datasets/GokuScraper/seedance-2-prompts-datasets/resolve/main/metadata.jsonl"

response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
content = response.text # Get the content as a string

df = pd.read_json(io.StringIO(content), lines=True)

print(f"✅ Loaded {len(df)} structured video prompts!")

✅ Loaded 8344 structured video prompts!


In [2]:
df["duration"] = df["spec"].apply(lambda x: x.get("duration"))
df["ratio"] = df["spec"].apply(lambda x: x.get("ratio"))
df["width"] = df["spec"].apply(lambda x: x.get("width"))
df["height"] = df["spec"].apply(lambda x: x.get("height"))

In [3]:
df['duration'].head()

,duration
0,15.10
1,15.13
2,15.12
3,15.10
4,15.13


In [4]:
from datasets import Dataset

ds = Dataset.from_pandas(df)
ds

Dataset({
    features: ['category', 'date', 'file_name', 'i18n', 'id', 'is_featured', 'media', 'model_info', 'platform', 'raw_p', 'slug', 'sourceLink', 'spec', 'version', 'duration', 'ratio', 'width', 'height'],
    num_rows: 8344
})

In [5]:
ds.features

{'category': Value('string'),
 'date': Value('timestamp[ns]'),
 'file_name': Value('string'),
 'i18n': {'en': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))},
  'zh': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))}},
 'id': Value('string'),
 'is_featured': Value('bool'),
 'media': {'c': Value('string'),
  'ref_images': List(Value('string')),
  'v': Value('string')},
 'model_info': {'model': Value('string'),
  'name': Value('string'),
  'version': Value('string')},
 'platform': Value('string'),
 'raw_p': Value('string'),
 'slug': Value('string'),
 'sourceLink': Value('string'),
 'spec': {'duration': Value('float64'),
  'height': Value('int64'),
  'ratio': Value('float64'),
  'safety_rating': Value('string'),
  'width': Value('int64')},
 'version': Value('int64'),
 'duration': Value('float64'),
 'ratio': Value('float64'),
 'width': Value('int64'),
 'height': Value('int64')}

In [6]:
ds['spec']['height']

Column([720, 720, 720, 720, 720])

In [7]:
# Map the dataset to our needs

def convert(example):
    en = example["i18n"].get("en", {})
    spec = example["spec"]

    prompt = en.get("p") or example.get("raw_p") or ""

    return {
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Write a Seedance 2 cinematic prompt.\n"
                    f"Category: {example['category']}\n"
                    f"Duration: {spec['duration']}\n"
                    f"Aspect Ratio: {spec['ratio']}"
                ),
            },
            {
                "role": "assistant",
                "content": prompt,
            },
        ]
    }

dataset = ds.map(convert)

Map:   0%|          | 0/8344 [00:00<?, ? examples/s]

In [8]:
# The part of the dataset that contains empty columns cannot be properly parsed by the "tokenize" function we're about to write

for i, ex in enumerate(dataset):
    for msg in ex["messages"]:
        if msg["content"] is None:
            print(i)
            print(ex)
            break

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct'
)

# The 'tokenize' function I was writing about before
def tokenize(example):
  text = tokenizer.apply_chat_template(
      example['messages'],
      tokenize=False,
      add_generation_prompt=False
  )

  return tokenizer(
      text,
      truncation=True,
      max_length=4096
  )

dataset_f = dataset.map(tokenize)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/8344 [00:00<?, ? examples/s]

In [ ]:
!accelerate config

In [11]:
!pip3 install deepspeed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.0 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.3-py3-none-any.whl size=1981996 sha256=85804b6623268fa3afb409c5adf53e7ffd546940bde50eef033c892b794280b8
  Stored in directory: /root/.cache/pip/wheels/ad/b5/17/8a0622bdb51b91b9fe37b9d6026f26e58a288bec0835a0c658
Successfully built deepspeed


In [3]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    device_map='auto',
    torch_dtype='bfloat16'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [1]:
!pip install --upgrade torchao

In [4]:
# Add LoRA
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [14]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_f,
    args=SFTConfig(
        output_dir="./seedance-with-lora",
        learning_rate=2e-4,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        packing=False,
    ),
)

trainer.train()

Truncating train dataset:   0%|          | 0/8344 [00:00<?, ? examples/s]

NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

In [16]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 15.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 41.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [17]:
import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer, SFTConfig

# --------------------------------------------------
# Free GPU memory (important in notebooks)
# --------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

# --------------------------------------------------
# Model
# --------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# --------------------------------------------------
# Prepare for QLoRA
# --------------------------------------------------

model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()

model.config.use_cache = False

# --------------------------------------------------
# LoRA
# --------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# --------------------------------------------------
# Trainer
# --------------------------------------------------

training_args = SFTConfig(
    output_dir="./seedance-lora",

    learning_rate=2e-4,

    num_train_epochs=3,

    per_device_train_batch_size=1,

    gradient_accumulation_steps=16,

    logging_steps=10,

    save_steps=500,

    max_length=1024,

    packing=False,

    fp16=True,

    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset_f,
    args=training_args,
)

trainer.train()

# --------------------------------------------------
# Save
# --------------------------------------------------

trainer.save_model("./seedance-lora")
tokenizer.save_pretrained("./seedance-lora")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


Truncating train dataset:   0%|          | 0/8344 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


AssertionError: No inf checks were recorded prior to update.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')
tokenizer.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')

In [ ]:
# Load model example
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = PeftModel.from_pretrained(
    base,
    "your-username/seedance-qwen2.5-7b-lora"
)

tokenizer = AutoTokenizer.from_pretrained("your-username/seedance-qwen2.5-7b-lora")